In [10]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import librosa
import os
from IPython.display import Audio
import matplotlib.pyplot as plt
import soundfile as sf

In [11]:
from models.tts.gpt_tts.gpt_tts import GPTTTS
from models.tts.gpt_tts.g2p_old_en import process, PHPONE2ID
# from g2p_en import G2p
from models.codec.codec_latent.codec_latent import LatentCodecEncoder, LatentCodecDecoderWithTimbre
from models.codec.amphion_codec.codec import CodecEncoder, CodecDecoder
from utils.util import load_config

In [12]:
cfg = load_config("egs/tts/LaCoTTS/exp_config_base.json")
print(cfg)

{'model_type': 'GPTTTS', 'dataset': ['Your Dataset Name'], 'preprocess': {'hop_size': 200, 'sample_rate': 16000, 'processed_dir': '', 'valid_file': 'valid.json', 'train_file': 'train.json'}, 'model': {'latent_codec': {'encoder': {'d_mel': 128, 'd_model': 96, 'num_blocks': 4, 'out_channels': 256, 'use_tanh': False, 'pretrained_ckpt': 'ckpts/latent_codec/latent_codec_enc.bin'}, 'decoder': {'in_channels': 256, 'num_quantizers': 1, 'codebook_size': 8192, 'codebook_dim': 8, 'quantizer_type': 'fvq', 'use_l2_normlize': True, 'vocos_dim': 512, 'vocos_intermediate_dim': 4096, 'vocos_num_layers': 16, 'ln_before_vq': True, 'use_pe': False, 'pretrained_ckpt': 'ckpts/latent_codec/latent_codec_dec.bin'}}, 'wav_codec': {'encoder': {'d_model': 96, 'out_channels': 128, 'up_ratios': [2, 4, 5, 5], 'use_tanh': False, 'pretrained_ckpt': 'ckpts/wav_codec/wav_codec_enc.bin'}, 'decoder': {'in_channels': 128, 'upsample_initial_channel': 1536, 'num_quantizers': 8, 'codebook_size': 1024, 'codebook_dim': 128, 'qu

# Audio Tokenizer: Convert Speech to Token

In [13]:
wav_codec_enc = CodecEncoder(
    cfg=cfg.model.wav_codec.encoder
)
wav_codec_dec = CodecDecoder(
    cfg=cfg.model.wav_codec.decoder
)

In [14]:
latent_codec_enc = LatentCodecEncoder(
    cfg=cfg.model.latent_codec.encoder
)
latent_codec_dec = LatentCodecDecoderWithTimbre(
    cfg=cfg.model.latent_codec.decoder
)

In [15]:
wav_codec_enc.load_state_dict(torch.load("ckpts/wav_codec/wav_codec_enc.bin"))
wav_codec_dec.load_state_dict(torch.load("ckpts/wav_codec/wav_codec_dec.bin"))
latent_codec_enc.load_state_dict(torch.load("ckpts/latent_codec/latent_codec_enc.bin"))
latent_codec_dec.load_state_dict(torch.load("ckpts/latent_codec/latent_codec_dec.bin"))

wav_codec_enc.eval()
wav_codec_dec.eval()
latent_codec_enc.eval()
latent_codec_dec.eval()

# wav_codec_enc.cuda()
# wav_codec_dec.cuda()
# latent_codec_enc.cuda()
# latent_codec_dec.cuda()

# requires_grad false
wav_codec_enc.requires_grad_(False)
wav_codec_dec.requires_grad_(False)
latent_codec_enc.requires_grad_(False)
latent_codec_dec.requires_grad_(False)

LatentCodecDecoderWithTimbre(
  (quantizer): ResidualVQ(
    (quantizers): ModuleList(
      (0): FactorizedVectorQuantize(
        (in_project): Conv1d(256, 8, kernel_size=(1,), stride=(1,))
        (out_project): Conv1d(8, 256, kernel_size=(1,), stride=(1,))
        (codebook): Embedding(8192, 8)
      )
    )
  )
  (model): Sequential(
    (0): VocosBackbone(
      (embed): Conv1d(256, 512, kernel_size=(7,), stride=(1,), padding=(3,))
      (norm): LayerNorm((512,), eps=1e-06, elementwise_affine=True)
      (convnext): ModuleList(
        (0-15): 16 x ConvNeXtBlock(
          (dwconv): Conv1d(512, 512, kernel_size=(7,), stride=(1,), padding=(3,), groups=512)
          (norm): LayerNorm((512,), eps=1e-06, elementwise_affine=True)
          (pwconv1): Linear(in_features=512, out_features=4096, bias=True)
          (act): GELU(approximate='none')
          (pwconv2): Linear(in_features=4096, out_features=512, bias=True)
        )
      )
      (final_layer_norm): LayerNorm((512,), eps=

# Latent Codec Language Model TTS

In [7]:
gpt_tts = GPTTTS(cfg=cfg.model.gpt_tts)
gpt_tts.load_state_dict(torch.load("./exps/latent_codec_gpt_tts/checkpoint/epoch-0020_step-0094800_loss-7.171329/pytorch_model.bin", map_location="cuda"))

<All keys matched successfully>

In [8]:
gpt_tts.eval()
# gpt_tts.cuda()
gpt_tts.requires_grad_(False)

GPTTTS(
  (model): LlamaForCausalLM(
    (model): LlamaModel(
      (embed_tokens): Embedding(8866, 1024, padding_idx=8858)
      (layers): ModuleList(
        (0-11): 12 x LlamaDecoderLayer(
          (self_attn): LlamaAttention(
            (q_proj): Linear(in_features=1024, out_features=1024, bias=False)
            (k_proj): Linear(in_features=1024, out_features=1024, bias=False)
            (v_proj): Linear(in_features=1024, out_features=1024, bias=False)
            (o_proj): Linear(in_features=1024, out_features=1024, bias=False)
            (rotary_emb): LlamaRotaryEmbedding()
          )
          (mlp): LlamaMLP(
            (gate_proj): Linear(in_features=1024, out_features=4096, bias=False)
            (up_proj): Linear(in_features=1024, out_features=4096, bias=False)
            (down_proj): Linear(in_features=4096, out_features=1024, bias=False)
            (act_fn): SiLU()
          )
          (input_layernorm): LlamaRMSNorm()
          (post_attention_layernorm): Llama

# Inference

In [42]:
source_text = "人们保留资格"
target_text = "团结产生力量"
source_wav = librosa.load("examples/ref/211.wav", sr=16000)[0]
Audio(source_wav, rate=16000)

In [43]:
# text tokenize
text = source_text + "," + target_text
# g2p = G2p()
# txt_struct, txt = process(text, g2p)
# phone_seq = [p for w in txt_struct for p in w[1]]
# phone_id = [PHPONE2ID[p] for p in phone_seq]
# phone_id = torch.LongTensor(phone_id).unsqueeze(0).cuda()
# phone_id.shape

In [44]:
from utils.g2p.g2p import phonemizer_g2p
ipa, phone = phonemizer_g2p(text, 'zh')
phone = torch.tensor(phone, dtype=torch.long)
phone_id = phone.unsqueeze(0)
ipa, phone_id

('ʐ əɜ n _ m ə1 n _ p ɑu2 _ l iouɜ _ ts i̪5 _ k oɜ _ th uaɜ n _ tɕ iɛɜ _ ts.h a2 n _ s. ə5 ŋ _ l i5 _ l iɑ5 ŋ ',
 tensor([[ 84, 142,  11,   1,  10, 138,  11,   1,  12, 208,   1,   9, 219,   1,
          113, 169,   1,   8, 109,   1, 112, 184,  11,   1, 114, 166,   1, 224,
           87,  11,   1, 111, 140,  23,   1,   9,  97,   1,   9, 161,  23]]))

In [45]:
# speech tokenize
prompt_wav = torch.FloatTensor(source_wav).unsqueeze(0)
# wav to latent
vq_emb = wav_codec_enc(prompt_wav.unsqueeze(1))
vq_emb = latent_codec_enc(vq_emb)
# latent to token
(
    _,
    vq_indices,
    _,
    _,
    _,
    speaker_embedding,
) = latent_codec_dec(vq_emb, vq=True, eval_vq=False, return_spk_embs=True)
prompt_id = vq_indices[0,:,:]
prompt_id.shape

torch.Size([1, 100])

In [46]:
code = prompt_id[0].cpu().numpy().tolist()
code = [int(c) for c in code]
code = np.array(code, dtype=np.int16)
# save code as small as possible
code.tofile("examples/ref/211.code")
# compute the size of the code
print(os.path.getsize("examples/ref/211.code"))
print(os.path.getsize("examples/ref/211.wav"))

200
40294


In [47]:
gen_tokens = gpt_tts.sample_hf(
    phone_id,
    prompt_id,
    max_length=3600,
    temperature=0.9,
    top_k=8192,
    top_p=0.95,
    repeat_penalty=1.0,
    classifer_free_guidance=0,   # i find speech speed will be faster if we set cfg > 1.0; if you don't want it, set cfg = 1.0
)

In [48]:
# gen token to latent
vq_post_emb = latent_codec_dec.vq2emb(gen_tokens.unsqueeze(0))
recovered_latent = latent_codec_dec(
    vq_post_emb, vq=False, speaker_embedding=speaker_embedding
)

In [49]:
# reconvered latent to wav
recovered_audio = wav_codec_dec(recovered_latent, vq=False)

In [50]:
sf.write("examples/recon/211.wav", recovered_audio.squeeze().cpu().numpy(), 16000)
Audio("examples/recon/211.wav", rate=16000)

# Inference_batch

In [17]:
import json
import time
from utils.g2p.g2p import phonemizer_g2p
floder_path = 'Wave16k16bNormalized'

# load json file
with open('./librispeech_ref_dur_3_test_full_with_punc_wdata.json', 'r') as f:
    json_data = f.read()
data = json.loads(json_data)
test_data = data["test_cases"]

for wav_info in test_data:
    
    output_file_name = wav_info["uid"] + ".wav"
    output_path = os.path.join("./gen_data", output_file_name)

    if os.path.exists(output_path):
        continue
    start = time.time()
    
    source_text = wav_info["text"]
    target_text = wav_info["target_text"]
    text = source_text + " " + target_text
    wav_path = wav_info["wav_path"].split('/')[-1]
    speaker_wav = os.path.join(floder_path, wav_path)
    source_wav = librosa.load(speaker_wav, sr=16000)[0]
    
    
    ipa, phone = phonemizer_g2p(text, 'en')
    phone = torch.tensor(phone, dtype=torch.long)
    phone_id = phone.unsqueeze(0)

    # speech tokenize
    prompt_wav = torch.FloatTensor(source_wav).unsqueeze(0)
    # wav to latent
    vq_emb = wav_codec_enc(prompt_wav.unsqueeze(1))
    vq_emb = latent_codec_enc(vq_emb)
    # latent to token
    (   _,
        vq_indices,
        _,
        _,
        _,
        speaker_embedding,
    ) = latent_codec_dec(vq_emb, vq=True, eval_vq=False, return_spk_embs=True)
    prompt_id = vq_indices[0,:,:]

    code = prompt_id[0].cpu().numpy().tolist()
    code = [int(c) for c in code]
    code = np.array(code, dtype=np.int16)
    # save code as small as possible
    code.tofile("examples/{}/{}.code".format(floder_path, wav_info["uid"]))


    gen_tokens = gpt_tts.sample_hf(
        phone_id,
        prompt_id,
        max_length=3600,
        temperature=0.9,
        top_k=8192,
        top_p=0.95,
        repeat_penalty=1.0,
        classifer_free_guidance=0,   # i find speech speed will be faster if we set cfg > 1.0; if you don't want it, set cfg = 1.0
    )


    # gen token to latent
    vq_post_emb = latent_codec_dec.vq2emb(gen_tokens.unsqueeze(0))
    recovered_latent = latent_codec_dec(
        vq_post_emb, vq=False, speaker_embedding=speaker_embedding
    )

    # reconvered latent to wav
    recovered_audio = wav_codec_dec(recovered_latent, vq=False)

    
    sf.write(output_path, recovered_audio.squeeze().cpu().numpy(), 16000)
                   
    rtf = (time.time() - start)
    print(f"RTF = {rtf:5f}")

RTF = 28.481703
RTF = 32.431969
RTF = 16.263288
RTF = 20.763110
RTF = 24.752709


KeyboardInterrupt: 

# Inference_batch WenetSpeech

In [13]:
import csv
import time
import random
from utils.g2p.g2p import phonemizer_g2p
floder_path = 'chinese_samples'


with open('./chinese_samples/samples.csv', newline='') as csvfile:
    csv_reader = csv.DictReader(csvfile)

    for row in csv_reader:
        start = time.time()
        
        text = row['transcript']
        # 获取 "target_path" 列的值
        wav_path = row['target_path'].split('/')[-1]
        speaker_wav = os.path.join(floder_path, wav_path)
        source_wav = librosa.load(speaker_wav, sr=16000)[0][:random.randint(16000, 32000)]
        
        
        ipa, phone = phonemizer_g2p(text, 'zh')
        phone = torch.tensor(phone, dtype=torch.long)
        phone_id = phone.unsqueeze(0)
    
        # speech tokenize
        prompt_wav = torch.FloatTensor(source_wav).unsqueeze(0)
        # wav to latent
        vq_emb = wav_codec_enc(prompt_wav.unsqueeze(1))
        vq_emb = latent_codec_enc(vq_emb)
        # latent to token
        (   _,
            vq_indices,
            _,
            _,
            _,
            speaker_embedding,
        ) = latent_codec_dec(vq_emb, vq=True, eval_vq=False, return_spk_embs=True)
        prompt_id = vq_indices[0,:,:]
    
        code = prompt_id[0].cpu().numpy().tolist()
        code = [int(c) for c in code]
        code = np.array(code, dtype=np.int16)
        # save code as small as possible
        code.tofile("examples/{}/{}.code".format(floder_path, wav_path.split(".")[0]))
    
    
        gen_tokens = gpt_tts.sample_hf(
            phone_id,
            prompt_id,
            max_length=3600,
            temperature=0.9,
            top_k=8192,
            top_p=0.95,
            repeat_penalty=1.0,
            classifer_free_guidance=0,   # i find speech speed will be faster if we set cfg > 1.0; if you don't want it, set cfg = 1.0
        )
    
    
        # gen token to latent
        vq_post_emb = latent_codec_dec.vq2emb(gen_tokens.unsqueeze(0))
        recovered_latent = latent_codec_dec(
            vq_post_emb, vq=False, speaker_embedding=speaker_embedding
        )
    
        # reconvered latent to wav
        recovered_audio = wav_codec_dec(recovered_latent, vq=False)
        output_file_name = wav_path
        output_path = os.path.join("./gen_data", output_file_name)
        
        sf.write(output_path, recovered_audio.squeeze().cpu().numpy(), 16000)
                       
        rtf = (time.time() - start)
        print(f"RTF = {rtf:5f}")

RTF = 6.131162
RTF = 15.145817
RTF = 10.999829
RTF = 20.836577
RTF = 23.686481
RTF = 21.318763
RTF = 10.174889
RTF = 25.140654
RTF = 18.273255
RTF = 123.198260
RTF = 67.029823
RTF = 16.049742
RTF = 35.390673
RTF = 14.876325
RTF = 46.820080
RTF = 14.445268
RTF = 9.186940
RTF = 7.795844
RTF = 45.679504
RTF = 9.838892
RTF = 16.581893
RTF = 47.353542
RTF = 29.331051
RTF = 28.016045
RTF = 10.123110
RTF = 19.507029
RTF = 108.574069
RTF = 29.031068
RTF = 32.454201
RTF = 14.793678
RTF = 11.766381
RTF = 15.484100
RTF = 63.101125
RTF = 60.546159
RTF = 66.088602
RTF = 5.906999
RTF = 9.173449
RTF = 31.004406
RTF = 44.785967
RTF = 12.839669
RTF = 12.340366
RTF = 32.068544
RTF = 18.755993
RTF = 10.386432
RTF = 30.199824
RTF = 34.074562
RTF = 22.216043
RTF = 47.251027
RTF = 9.484639
RTF = 27.197513
RTF = 28.885927
RTF = 66.810517
RTF = 26.291743
RTF = 37.564239
RTF = 99.405272
RTF = 45.241912
RTF = 61.980919
RTF = 6.317306
RTF = 24.077532
RTF = 75.127301
RTF = 15.595843
RTF = 39.798730


IndexError: index out of range in self